# Job Requisition Validation

This notebook validates synthetic job requisitions.

The requisition table includes:

- Historical filled requisitions
- Cancelled requisitions
- Current open requisitions
- Hiring departments, job roles, and locations
- Target headcounts
- Recruiters responsible for each opening

The main rules are:

- Requisition IDs are complete, unique, and sequential.
- Foreign keys reference valid records.
- Target headcount is a positive whole number.
- Open requisitions have no close date.
- Filled and cancelled requisitions have close dates.
- Recruiters are employed throughout the requisition period.
- Filled requisitions cover all employee hiring groups.
- Filled target headcount sums to 10,000.

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"


employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

departments = pd.read_csv(
    RAW_DATA_DIR / "departments.csv"
)

locations = pd.read_csv(
    RAW_DATA_DIR / "locations.csv"
)

job_roles = pd.read_csv(
    RAW_DATA_DIR / "job_roles.csv"
)

job_requisitions = pd.read_csv(
    RAW_DATA_DIR / "job_requisitions.csv",
    parse_dates=[
        "open_date",
        "close_date",
    ],
)


print(
    "Employees:",
    employees.shape,
)

print(
    "Job requisitions:",
    job_requisitions.shape,
)

Employees: (10000, 15)
Job requisitions: (2779, 9)


In [2]:
job_requisitions.head(10)

,requisition_id,job_role_id,department_id,location_id,open_date,close_date,target_headcount,recruiter_id,requisition_status
0,300001,1,1,1,2021-01-01,2021-03-10,3,107620,Filled
1,300002,1,1,3,2021-01-01,2021-01-07,1,107948,Filled
2,300003,1,1,4,2021-01-01,2021-03-10,3,107620,Filled
3,300004,1,1,5,2021-01-01,2021-03-07,4,107948,Filled
4,300005,2,1,1,2021-01-01,2021-03-28,5,107948,Filled
5,300006,2,1,3,2021-01-01,2021-03-30,7,107948,Filled
6,300007,2,1,4,2021-01-01,2021-03-31,3,108065,Filled
7,300008,3,1,2,2021-01-01,2021-03-29,3,108065,Filled
8,300009,3,1,3,2021-01-01,2021-03-25,5,108065,Filled
9,300010,3,1,5,2021-01-01,2021-01-30,4,108065,Filled


In [3]:
job_requisitions.info()

<class 'pandas.DataFrame'>
RangeIndex: 2779 entries, 0 to 2778
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   requisition_id      2779 non-null   int64         
 1   job_role_id         2779 non-null   int64         
 2   department_id       2779 non-null   int64         
 3   location_id         2779 non-null   int64         
 4   open_date           2779 non-null   datetime64[us]
 5   close_date          2659 non-null   datetime64[us]
 6   target_headcount    2779 non-null   int64         
 7   recruiter_id        2779 non-null   int64         
 8   requisition_status  2779 non-null   str           
dtypes: datetime64[us](2), int64(6), str(1)
memory usage: 195.5 KB


In [4]:
expected_columns = [
    "requisition_id",
    "job_role_id",
    "department_id",
    "location_id",
    "open_date",
    "close_date",
    "target_headcount",
    "recruiter_id",
    "requisition_status",
]

expected_ids = list(
    range(
        300_001,
        300_001
        + len(job_requisitions),
    )
)

structure_checks = pd.Series(
    {
        "table has nine columns": (
            job_requisitions.columns.tolist()
            == expected_columns
        ),
        "requisition IDs are complete": (
            job_requisitions[
                "requisition_id"
            ].notna().all()
        ),
        "requisition IDs are unique": (
            job_requisitions[
                "requisition_id"
            ].is_unique
        ),
        "requisition IDs are sequential": (
            job_requisitions[
                "requisition_id"
            ].astype(int).tolist()
            == expected_ids
        ),
        "target headcount is positive": (
            job_requisitions[
                "target_headcount"
            ].ge(1).all()
        ),
        "target headcount uses whole numbers": (
            job_requisitions[
                "target_headcount"
            ]
            .astype(float)
            .mod(1)
            .eq(0)
            .all()
        ),
    },
    name="passed",
)

structure_checks

table has nine columns                 True
requisition IDs are complete           True
requisition IDs are unique             True
requisition IDs are sequential         True
target headcount is positive           True
target headcount uses whole numbers    True
Name: passed, dtype: bool

In [5]:
foreign_key_checks = pd.Series(
    {
        "job role IDs are valid": (
            set(
                job_requisitions[
                    "job_role_id"
                ]
            )
            .issubset(
                set(
                    job_roles[
                        "job_role_id"
                    ]
                )
            )
        ),
        "department IDs are valid": (
            set(
                job_requisitions[
                    "department_id"
                ]
            )
            .issubset(
                set(
                    departments[
                        "department_id"
                    ]
                )
            )
        ),
        "location IDs are valid": (
            set(
                job_requisitions[
                    "location_id"
                ]
            )
            .issubset(
                set(
                    locations[
                        "location_id"
                    ]
                )
            )
        ),
        "recruiter IDs are valid": (
            set(
                job_requisitions[
                    "recruiter_id"
                ]
            )
            .issubset(
                set(
                    employees[
                        "employee_id"
                    ]
                )
            )
        ),
    },
    name="passed",
)

foreign_key_checks

job role IDs are valid      True
department IDs are valid    True
location IDs are valid      True
recruiter IDs are valid     True
Name: passed, dtype: bool

In [6]:
open_requisitions = (
    job_requisitions[
        job_requisitions[
            "requisition_status"
        ]
        == "Open"
    ]
)

filled_requisitions = (
    job_requisitions[
        job_requisitions[
            "requisition_status"
        ]
        == "Filled"
    ]
)

cancelled_requisitions = (
    job_requisitions[
        job_requisitions[
            "requisition_status"
        ]
        == "Cancelled"
    ]
)

closed_requisitions = (
    job_requisitions[
        job_requisitions[
            "requisition_status"
        ]
        .isin(
            [
                "Filled",
                "Cancelled",
            ]
        )
    ]
)

status_and_date_checks = pd.Series(
    {
        "statuses are valid": (
            set(
                job_requisitions[
                    "requisition_status"
                ]
            )
            .issubset(
                {
                    "Open",
                    "Filled",
                    "Cancelled",
                }
            )
        ),
        "there are 120 open requisitions": (
            len(open_requisitions)
            == 120
        ),
        "there are 250 cancelled requisitions": (
            len(cancelled_requisitions)
            == 250
        ),
        "open requisitions have no close date": (
            open_requisitions[
                "close_date"
            ].isna().all()
        ),
        "closed requisitions have close dates": (
            closed_requisitions[
                "close_date"
            ].notna().all()
        ),
        "close dates follow open dates": (
            closed_requisitions[
                "close_date"
            ]
            .ge(
                closed_requisitions[
                    "open_date"
                ]
            )
            .all()
        ),
    },
    name="passed",
)

status_and_date_checks

statuses are valid                      True
there are 120 open requisitions         True
there are 250 cancelled requisitions    True
open requisitions have no close date    True
closed requisitions have close dates    True
close dates follow open dates           True
Name: passed, dtype: bool

In [7]:
recruiter_details = (
    employees[
        [
            "employee_id",
            "hire_date",
            "termination_date",
        ]
    ]
    .rename(
        columns={
            "employee_id": "recruiter_id",
            "hire_date": "recruiter_hire_date",
            "termination_date": (
                "recruiter_termination_date"
            ),
        }
    )
)

requisition_details = (
    job_requisitions
    .merge(
        recruiter_details,
        on="recruiter_id",
        how="left",
    )
)

requisition_details[
    "requisition_end_date"
] = (
    requisition_details[
        "close_date"
    ]
    .fillna(
        pd.Timestamp("2026-06-30")
    )
)

recruiter_checks = pd.Series(
    {
        "recruiters were hired before openings": (
            requisition_details[
                "recruiter_hire_date"
            ]
            .le(
                requisition_details[
                    "open_date"
                ]
            )
            .all()
        ),
        "recruiters remained through requisition end": (
            (
                requisition_details[
                    "recruiter_termination_date"
                ].isna()
                | requisition_details[
                    "recruiter_termination_date"
                ].ge(
                    requisition_details[
                        "requisition_end_date"
                    ]
                )
            )
            .all()
        ),
    },
    name="passed",
)

recruiter_checks

recruiters were hired before openings          True
recruiters remained through requisition end    True
Name: passed, dtype: bool

In [8]:
employee_hiring_groups = (
    employees.copy()
)

employee_hiring_groups[
    "hire_quarter"
] = (
    employee_hiring_groups[
        "hire_date"
    ].dt.to_period("Q")
)

expected_hiring_groups = (
    employee_hiring_groups
    .groupby(
        [
            "job_role_id",
            "department_id",
            "location_id",
            "hire_quarter",
        ],
        as_index=False,
    )
    .agg(
        expected_headcount=(
            "employee_id",
            "size",
        ),
        earliest_hire_date=(
            "hire_date",
            "min",
        ),
        latest_hire_date=(
            "hire_date",
            "max",
        ),
    )
)

filled_group_details = (
    filled_requisitions.copy()
)

filled_group_details[
    "hire_quarter"
] = (
    filled_group_details[
        "close_date"
    ].dt.to_period("Q")
)

group_keys = [
    "job_role_id",
    "department_id",
    "location_id",
    "hire_quarter",
]

hiring_group_alignment = (
    expected_hiring_groups
    .merge(
        filled_group_details[
            group_keys
            + [
                "open_date",
                "close_date",
                "target_headcount",
            ]
        ],
        on=group_keys,
        how="outer",
        indicator=True,
    )
)

filled_checks = pd.Series(
    {
        "every hiring group has a filled requisition": (
            hiring_group_alignment[
                "_merge"
            ].eq("both").all()
        ),
        "filled headcounts match employee groups": (
            hiring_group_alignment[
                "target_headcount"
            ]
            .eq(
                hiring_group_alignment[
                    "expected_headcount"
                ]
            )
            .all()
        ),
        "filled requisitions open before first hire": (
            hiring_group_alignment[
                "open_date"
            ]
            .le(
                hiring_group_alignment[
                    "earliest_hire_date"
                ]
            )
            .all()
        ),
        "filled close dates match last hires": (
            hiring_group_alignment[
                "close_date"
            ]
            .eq(
                hiring_group_alignment[
                    "latest_hire_date"
                ]
            )
            .all()
        ),
        "filled target headcount totals 10,000": (
            filled_requisitions[
                "target_headcount"
            ].sum()
            == 10_000
        ),
    },
    name="passed",
)

filled_checks

every hiring group has a filled requisition    True
filled headcounts match employee groups        True
filled requisitions open before first hire     True
filled close dates match last hires            True
filled target headcount totals 10,000          True
Name: passed, dtype: bool

In [9]:
requisition_summary = (
    job_requisitions
    .groupby(
        "requisition_status"
    )
    .agg(
        requisition_count=(
            "requisition_id",
            "count",
        ),
        total_target_headcount=(
            "target_headcount",
            "sum",
        ),
        average_target_headcount=(
            "target_headcount",
            "mean",
        ),
        average_days_open=(
            "open_date",
            lambda values: None,
        ),
    )
)

requisition_summary = (
    requisition_summary.drop(
        columns=[
            "average_days_open"
        ]
    )
)

requisition_summary[
    "average_target_headcount"
] = (
    requisition_summary[
        "average_target_headcount"
    ].round(2)
)

requisition_summary

,requisition_count,total_target_headcount,average_target_headcount
requisition_status,,,
Cancelled,250,921,3.68
Filled,2409,10000,4.15
Open,120,542,4.52


In [10]:
closed_duration_summary = (
    closed_requisitions
    .assign(
        days_open=(
            closed_requisitions[
                "close_date"
            ]
            - closed_requisitions[
                "open_date"
            ]
        ).dt.days
    )
    .groupby(
        "requisition_status"
    )
    .agg(
        average_days_open=(
            "days_open",
            "mean",
        ),
        minimum_days_open=(
            "days_open",
            "min",
        ),
        maximum_days_open=(
            "days_open",
            "max",
        ),
    )
    .round(2)
)

closed_duration_summary

,average_days_open,minimum_days_open,maximum_days_open
requisition_status,,,
Cancelled,49.99,10,90
Filled,93.04,2,165


In [11]:
all_checks = pd.concat(
    [
        structure_checks,
        foreign_key_checks,
        status_and_date_checks,
        recruiter_checks,
        filled_checks,
    ]
)

validation_results = pd.DataFrame(
    {
        "check": all_checks.index,
        "passed": all_checks.values,
    }
)

validation_results

,check,passed
0,table has nine columns,True
1,requisition IDs are complete,True
2,requisition IDs are unique,True
3,requisition IDs are sequential,True
4,target headcount is positive,True
5,target headcount uses whole numbers,True
6,job role IDs are valid,True
7,department IDs are valid,True
8,location IDs are valid,True
9,recruiter IDs are valid,True


In [12]:
if validation_results["passed"].all():
    print(
        "All job-requisition validation "
        "checks passed."
    )
else:
    print(
        "One or more job-requisition "
        "validation checks failed."
    )

All job-requisition validation checks passed.


## Conclusions

The synthetic job-requisition table successfully represents historical and current recruiting demand.

### Successful checks

- Requisition IDs are complete, unique, and sequential.
- Job role, department, location, and recruiter foreign keys are valid.
- Target headcount values are positive whole numbers.
- Open requisitions have blank close dates.
- Filled and cancelled requisitions have valid close dates.
- Recruiters are employed throughout their assigned requisition periods.
- Filled requisitions cover every employee hiring group.
- Filled requisition headcounts match employee hiring groups.
- Filled target headcount totals 10,000.
- There are 120 current open requisitions.
- There are 250 cancelled requisitions.
- All job-requisition validation checks passed.

### Current simplifications

- Historical filled requisitions are grouped by hiring quarter.
- Each employee is represented in one filled requisition group.
- Cancelled and open requisitions use valid existing role, department, and location combinations.
- Recruiter assignments are synthetic.
- Open requisitions do not yet have applications.
- Candidate applications will be generated in the next checkpoint.